# Код для DeepSeek V3.2

In [ ]:
import pandas as pd
import requests
import json
import time
import os
import re
import csv
import random
from datetime import datetime
from dotenv import load_dotenv


def extract_json_from_text(text):
    if not text:
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # модель иногда оборачивает JSON в пояснения или markdown — ловим первую {...} группу
    match = re.search(r'\{[\s\S]*\}', text)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None
    return None


def log_progress(log_path: str, review_id, status: str, data=None, error: str = None):
    # jsonl, а не один большой json — чтобы можно было докидывать строчки и не перечитывать файл
    entry = {
        "ts": datetime.now().isoformat(timespec="seconds"),
        "review_id": review_id,
        "status": status,
    }
    if data is not None:
        entry["results_count"] = len(data.get("results", []))
        entry["results"] = data.get("results", [])
    if error is not None:
        entry["error"] = error
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


def load_processed_ids(log_path: str) -> set:
    # пробегаемся по логу и собираем id, на которых был статус ok — их повторно гонять не надо
    processed = set()
    if not os.path.exists(log_path):
        return processed
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                if entry.get("status") == "ok":
                    processed.add(entry["review_id"])
            except json.JSONDecodeError:
                pass
    return processed


load_dotenv()

GENAPI_API_KEY = os.getenv("GENAPI_API_KEY")
if not GENAPI_API_KEY:
    raise ValueError("GENAPI_API_KEY не найден. Проверьте файл .env")

INPUT_CSV      = "reviews.csv"
OUTPUT_CSV     = "results_with_metadata_deepseek.csv"
PROGRESS_LOG   = "progress_log.jsonl"

MAX_RETRIES      = 7
BASE_DELAY       = 3
MAX_DELAY        = 60
REQUEST_INTERVAL = 3.0
JITTER_MAX       = 1.5

URL_SUBMIT = "https://api.gen-api.ru/api/v1/networks/deepseek-chat"
URL_RESULT = "https://api.gen-api.ru/api/v1/request/get/{request_id}"

POLL_INTERVAL = 4.0
POLL_TIMEOUT  = 300

ID_COL   = "review_id"
TEXT_COL = "review_text"

SYSTEM_PROMPT = """Ты — аналитик отзывов об онлайн-курсах. Извлеки все мнения, аспектные термины, тональность и аспект строго по правилам. Работай строго по правилам ниже.

1.Точное цитирование — точные подстроки из review_text. Запрещен пересказ, синонимы, изменение формы слов.
2.Не пропускать оценочные фразы
3.Не извлекать лишние слова
4.Лучше разделить, чем объединить мнения
5. Опускать аспектное слово при извлечении мнения
6. Одно мнение = одна оценка = один аспектный термин = один аспект

ВХОД: review_id (int),  review_text (текст). Остальные поля игнорируй

ШАГ 1:
Аспектный термин — 1–4 слова, конкретный объект курса (лекции, задания, платформа, преподаватель и т.п.), к которому относится оценка.
—Выбирай минимальный: «сам контент» → «контент»
—Точная подстрока: ЗАПРЕЩЕНО изменять форму (падеж, число, время): «итоговую работу» НЕ равно «итоговая работа»
— НЕ местоимения, процессы/действия/абстракции без объекта, вводные фразы
— НОВЫЙ объект → новый аспектный термин

ШАГ 2:
Мнение — 2–10 слов, явная или косвенная оценка об объекте.
— Минимальная оценочная часть фразы с модификатором («очень», «довольно») при его наличии
— Точная подстрока: нельзя добавлять, удалять, обобщать, переформулировать слова
— Нельзя дублировать аспектное слово
— В пределах одного предложения
— Считать мнением: оценочные прилагательные/наречия/глаголы, устойчивые выражения («на высшем уровне»), скрытые оценки («длится 7 минут»), развернутые оценки («сформировался четкий план моего краткосрочного и долгосрочного развития»)
— НЕ считать мнением: нейтральные факты («было 12 модулей»), перечисления  тем/форматов без оценки.

РАЗДЕЛЯТЬ:
1.противопоставления (но, однако, зато): («материал хороший, но неструктурированный → «хороший» и «неструктурированный»
2.разные атрибуты одного объекта: «проверяют быстро, с полезными комментариями» → «быстро» (скорость) и «с полезными комментариями» (качество).
3.разные аспекты в одной фразе (например, Стоимость + Контент): «купила за 50 тыс руб, а за час прошла курс» → «купила за 50 тыс руб» (Цена); «за час прошла курс» (Контент)

НЕ РАЗДЕЛЯТЬ:
1.однородные члены без противопоставления с одним и тем же атрибутом объекта («отношение сотрудников и наставников очень доброе, понимающее→описано только качество отношения => единое мнение «очень доброе, понимающее»)

САМОПРОВЕРКА
— Аспектный термин: точная подстрока (1–4 слова), конкретный объект.
— Мнение: точная подстрока с оценкой.
— Все оценки извлечены, нет выдуманных, нет дублей.

ШАГ 3:
ТОНАЛЬНОСТЬ:
положительная — положительная оценка
отрицательная — отрицательная оценка
АСПЕКТЫ:
Преподаватели — работа преподавателей и лекторов (стиль подачи материала, компетентность, умение объяснять, вовлеченность, готовность отвечать на вопросы по содержанию)
Контент — теоретическое содержание курса (программа, темы, структура, полнота, актуальность, качество, объем информации, последовательность разделов/модулей, формат, необходимость поиска дополнительных материалов)
Практика — проекты, кейсы, задания и т.п., т.е. все что студент выполняет для отработки навыков (сложность, полезность, количество, наличие, связь с теорией)
Техническая поддержка — качество помощи от технической поддержки (скорость ответов, доброжелательность, компетентность)
Команда сопровождения — работа кураторов / координаторов (качество и наличие обратной связи по заданиям, скорость проверки, доброжелательность, решение организационных и учебных вопросов)
Сообщество – общение с одногруппниками, чаты, взаимопомощь, атмосфера в коллективе
Платформа — интерфейс, навигация, стабильность работы сайта/приложения
Стоимость — стоимость, скидки, рассрочка, соотношение цена/качество, возврат средств
Результаты — приобретенные после окончания курса навыки, трудоустройство после окончания курса, достижения после окончания курса, карьерный рост, портфолио
Сертификаты — наличие диплома или сертификата после обучения (ценность, процесс получения)
Гибкость учебного процесса — график, дедлайны, наличие записей лекций, возможность совмещать с работой и личной жизнью возможность обучаться в любое удобное время, возможность продления дедлайна
Дополнительно — бонусы, плюшки, стажировки, дополнительные активности, внеучебные активности, программа трудоустройства, составление резюме
Общее — общее впечатление без указания конкретных причин, если нельзя отнести к другим аспектам
3. Выбор аспекта
— Сначала ориентируйся на аспектный термин, затем на мнение
— Наиболее конкретный аспект

ФОРМАТ ВЫВОДА JSON
{
 "review_id": <id>,
 "results": [
 {
 "aspect_term": "точная подстрока из текста (1–4 слова)",
 "opinion": "точная подстрока из текста (оценочная часть)",
 "sentiment": "положительная | отрицательная ",
 "aspect": "из списка аспектов"
 }
 ]
}

ПРИМЕРЫ
[
 {
 "input": {
 "review_id": 3,
 " review_text": "Кураторы тоже большие молодцы — проверяют работы быстро, с конкретными и полезными комментариями."
 },
 "output": {
 "review_id": 3,
 "results": [
 {"aspect_term": "кураторы", "opinion": "большие молодцы", "sentiment": "положительная", "aspect": "Команда сопровождения"},
 {"aspect_term": "кураторы", "opinion": "проверяют работы быстро", "sentiment": "положительная", "aspect": "Команда сопровождения"},
 {"aspect_term": "кураторы", "opinion": "с конкретными и полезными комментариями", "sentiment": "положительная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 2,
 " review_text": "внутренняя атмосфера тепла, уюта, очень уважительного человеческого отношения!"
 },
 "output": {
 "review_id": 2,
 "results": [
 {"aspect_term": "внутренняя атмосфера", "opinion": "тепла, уюта, очень уважительного человеческого отношения", "sentiment": "положительная", "aspect": "Общее"}
 ]
 }
 },
 {
 "input": {
 "review_id": 30,
 " review_text": "На уроке «КАК повысить конверсию», нам рассказывают: ЧТО такое конверсия. Хочу отметить, что урок длится 7 минут."
 },
 "output": {
 "review_id": 30,
 "results": [
 {"aspect_term": "уроке «КАК повысить конверсию»", "opinion": "рассказывают: ЧТО такое конверсия", "sentiment": "отрицательная", "aspect": "Контент"}
 {"aspect_term": "урок", "opinion": "длится 7 минут", "sentiment": "отрицательная", "aspect": "Контент"}
 ]
 }
 },
 {
 "input": {
 "review_id": 6,
 " review_text": "Качественные материалы составлены таким образом, что вы не чувствуете обязанности пройти урок, вы испытываете удовольствие от получения новых знаний."
 },
 "output": {
 "review_id": 6,
 "results": [
 {"aspect_term": "материалы", "opinion": "качественные", "sentiment": "положительная", "aspect": "Контент"},
 {"aspect_term": "материалы", "opinion": "испытываете удовольствие от получения новых знаний", "sentiment": "положительная", "aspect": "Контент"}
 ]
 }
 },
 {
 "input": {
 "review_id": 14,
 " review_text": "Куратор не помощник, только отписчик."
 },
 "output": {
 "review_id": 14,
 "results": [
 {"aspect_term": "куратор", "opinion": "не помощник, только отписчик", "sentiment": "отрицательная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 13,
 " review_text": "Технические сбои бывают, но решаются быстро."
 },
 "output": {
 "review_id": 13,
 "results": [
 {"aspect_term": "технические сбои", "opinion": "бывают", "sentiment": "отрицательная", "aspect": "Платформа"},
 {"aspect_term": "технические сбои", "opinion": "решаются быстро", "sentiment": "положительная", "aspect": "Платформа"}
 ]
 }
 },
 {
 "input": {
 "review_id": 23,
 " review_text": "Отношение сотрудников и наставников очень доброе, понимающее и профессиональное."
 },
 "output": {
 "review_id": 23,
 "results": [
 {"aspect_term": "отношение сотрудников и наставников", "opinion": "очень доброе, понимающее и профессиональное", "sentiment": "положительная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 17,
 " review_text": "Теперь у меня есть диплом гос образца о профессиональной переподготовке."
 },
 "output": {
 "review_id": 17,
 "results": [
 {"aspect_term": "диплом гос образца", "opinion": "у меня есть", "sentiment": "положительная", "aspect": "Сертификаты"}
 ]
 }
 },
 {
 "input": {
 "review_id": 4,
 " review_text": "Лекции были примерно такие «Если сложить 2+2, то в итоге будет 4», в домашке к этой лекции было «Соберите ракету и отправьте ее в космос»."
 },
 "output": {
 "review_id": 4,
 "results": [
 {"aspect_term": "лекции", "opinion": " «Если сложить 2+2, то в итоге будет 4»", "sentiment": "отрицательная", "aspect": "Контент"},
 {"aspect_term": "домашке", "opinion": "было «Соберите ракету и отправьте ее в космос»", "sentiment": "отрицательная", "aspect": "Практика"}
 ]
 }
 },
 {
 "input": {
 "review_id": 12,
 " review_text": "Практика проектов по нейросетям проходила на учебной платформе, которая периодически зависала, или в приложении, где стабильность работы тоже оставляла желать лучшего."
 },
 "output": {
 "review_id": 12,
 "results": [
 {"aspect_term": "учебной платформе", "opinion": "периодически зависала", "sentiment": "отрицательная", "aspect": "Платформа"},
 {"aspect_term": "приложении", "opinion": "стабильность работы тоже оставляла желать лучшего", "sentiment": "отрицательная", "aspect": "Платформа"}
 ]
 }
 },
]
"""


def make_session() -> requests.Session:
    # пул из одного коннекта — gen-api не любит, когда мы держим много открытых
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        max_retries=0,
        pool_connections=1,
        pool_maxsize=1,
        pool_block=False,
    )
    session.mount("https://", adapter)
    session.headers.update({
        "Content-Type": "application/json",
        "Authorization": f"Bearer {GENAPI_API_KEY}",
        "Connection": "keep-alive",
    })
    return session


_session: requests.Session = None
_last_request_time: float = 0


def analyze_single_review(review_id, comment_text: str, log_path: str):
    # схема: submit -> получаем request_id -> опрашиваем GET до success.
    # синхронный режим тут не работает на длинных промптах — сервер рвёт TCP по таймауту
    global _session, _last_request_time

    if _session is None:
        _session = make_session()

    if not isinstance(comment_text, str):
        comment_text = str(comment_text)

    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"review_id: {review_id}\ncomment: {comment_text}"},
        ],
        "temperature": 0.1,
        "max_tokens": 3500,
        "response_format": {"type": "json_object"},
    }

    for attempt in range(MAX_RETRIES):
        # троттлинг: спим столько, чтобы между запросами было ~REQUEST_INTERVAL + рандом
        elapsed = time.time() - _last_request_time
        wait_base = REQUEST_INTERVAL - elapsed
        jitter = random.uniform(0, JITTER_MAX)
        if wait_base + jitter > 0:
            time.sleep(wait_base + jitter)

        try:
            resp = _session.post(URL_SUBMIT, json=payload, timeout=30)

            if resp.status_code == 429:
                # экспоненциальный бэкофф — единственное, что реально помогает против rate-limit
                wait = min(BASE_DELAY * (2 ** attempt), MAX_DELAY) + random.uniform(0, JITTER_MAX)
                print(f"   429 rate-limit -> ждём {wait:.1f}s (попытка {attempt+1}/{MAX_RETRIES})")
                log_progress(log_path, review_id, "retry",
                             error=f"429, wait={wait:.1f}s, attempt={attempt+1}")
                time.sleep(wait)
                continue

            if resp.status_code >= 500:
                wait = min(BASE_DELAY * (2 ** attempt), MAX_DELAY) + random.uniform(0, JITTER_MAX)
                print(f"   HTTP {resp.status_code} -> ждём {wait:.1f}s (попытка {attempt+1}/{MAX_RETRIES})")
                log_progress(log_path, review_id, "retry",
                             error=f"HTTP {resp.status_code}, attempt={attempt+1}")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            submit_json = resp.json()
            _last_request_time = time.time()

            # в зависимости от версии API ключ называется по-разному, страхуемся
            request_id = submit_json.get("id") or submit_json.get("request_id")
            if not request_id:
                print(f"   Нет request_id в ответе: {submit_json}")
                log_progress(log_path, review_id, "retry",
                             error=f"no_request_id, resp={str(submit_json)[:200]}, attempt={attempt+1}")
                time.sleep(BASE_DELAY + random.uniform(0, JITTER_MAX))
                continue

            print(f"   Запрос {request_id} принят, ждём результат...")

            poll_url = URL_RESULT.format(request_id=request_id)
            deadline = time.time() + POLL_TIMEOUT

            while time.time() < deadline:
                time.sleep(POLL_INTERVAL)
                try:
                    poll_resp = _session.get(poll_url, timeout=15)
                    poll_resp.raise_for_status()
                    poll_json = poll_resp.json()
                except Exception as poll_err:
                    # единичный сбой polling-а — не страшно, просто повторим на следующей итерации
                    print(f"   Ошибка polling: {poll_err}")
                    continue

                status = poll_json.get("status", "")

                if status == "processing":
                    progress = poll_json.get("progress", 0)
                    print(f"   Обработка... {progress}%")
                    continue

                if status == "error":
                    print(f"   Сервер вернул ошибку для {review_id}: {poll_json}")
                    log_progress(log_path, review_id, "retry",
                                 error=f"server_error: {str(poll_json)[:200]}, attempt={attempt+1}")
                    break

                if status == "success":
                    # путь до контента: result[0].message.content (как у openai-совместимых)
                    response_list = poll_json.get("result", poll_json.get("response", []))
                    if not response_list:
                        print(f"   Пустой result/response в результате: {poll_json}")
                        log_progress(log_path, review_id, "retry",
                                     error=f"empty_response_in_result, attempt={attempt+1}")
                        break

                    raw = response_list[0].get("message", {}).get("content", "")
                    # lstrip BOM — иногда прилетает в начале и ломает json.loads
                    answer_text = (raw if isinstance(raw, str) else str(raw)).strip().lstrip("\ufeff")
                    parsed = extract_json_from_text(answer_text)

                    if parsed:
                        log_progress(log_path, review_id, "ok", data=parsed)
                        return parsed

                    print(f"   Битый JSON для {review_id}")
                    print(f"   RAW: {answer_text[:300]}")
                    log_progress(log_path, review_id, "retry",
                                 error=f"bad_json, attempt={attempt+1}, raw={answer_text[:200]}")
                    break

            else:
                # while ... else: сработает, только если вышли по условию deadline, а не break-ом
                print(f"   Timeout polling для {review_id} (>{POLL_TIMEOUT}s)")
                log_progress(log_path, review_id, "retry",
                             error=f"poll_timeout, attempt={attempt+1}")

        except (requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError) as e:
            # сеть упала или сервер закрыл соединение — старую сессию выкидываем, делаем новую
            wait = min(BASE_DELAY * (2 ** attempt), MAX_DELAY) + random.uniform(0, JITTER_MAX)
            print(f"   Сетевая ошибка для {review_id}: {e} -> пересоздаём сессию, ждём {wait:.1f}s (попытка {attempt+1}/{MAX_RETRIES})")
            log_progress(log_path, review_id, "retry",
                         error=f"connection_error: {str(e)[:120]}, attempt={attempt+1}")
            try:
                _session.close()
            except Exception:
                pass
            _session = make_session()
            time.sleep(wait)

        except requests.exceptions.Timeout:
            wait = min(BASE_DELAY * (2 ** attempt), MAX_DELAY) + random.uniform(0, JITTER_MAX)
            print(f"   Timeout отправки для {review_id} -> ждём {wait:.1f}s (попытка {attempt+1}/{MAX_RETRIES})")
            log_progress(log_path, review_id, "retry",
                         error=f"submit_timeout, attempt={attempt+1}")
            time.sleep(wait)

        except Exception as e:
            wait = BASE_DELAY + random.uniform(0, JITTER_MAX)
            print(f"   Неизвестная ошибка для {review_id}: {e}")
            log_progress(log_path, review_id, "retry",
                         error=f"unknown: {str(e)[:120]}, attempt={attempt+1}")
            time.sleep(wait)

    log_progress(log_path, review_id, "error",
                 error=f"all {MAX_RETRIES} attempts failed")
    return None


def process_csv_with_metadata(input_path: str, output_path: str, log_path: str = PROGRESS_LOG):
    try:
        # QUOTE_ALL + escapechar, потому что в отзывах хватает кавычек и переносов внутри ячеек
        original_df = pd.read_csv(
            input_path,
            quoting=csv.QUOTE_ALL,
            escapechar="\\",
            on_bad_lines="skip",
        )
        print(f"Загружено {len(original_df)} отзывов.")
        print(f"Колонки: {list(original_df.columns)}")
    except FileNotFoundError:
        print(f"Файл {input_path} не найден.")
        return

    if ID_COL not in original_df.columns:
        raise ValueError(f"Колонка '{ID_COL}' не найдена в CSV.")
    if TEXT_COL not in original_df.columns:
        raise ValueError(f"Колонка '{TEXT_COL}' не найдена в CSV.")

    # всё, что не id и не текст — тащим в результат как есть (рейтинг, источник и т.п.)
    extra_cols = [c for c in original_df.columns if c not in [ID_COL, TEXT_COL]]
    print(f"Доп. колонки: {extra_cols}")

    processed_ids = load_processed_ids(log_path)
    if processed_ids:
        print(f"Найден лог прогресса — пропускаем {len(processed_ids)} уже обработанных отзывов.")

    # если прогон оборвался посередине, подбираем то, что уже было записано в CSV
    existing_rows = []
    if os.path.exists(output_path) and processed_ids:
        try:
            existing_df = pd.read_csv(output_path, quoting=csv.QUOTE_ALL,
                                      escapechar="\\", on_bad_lines="skip")
            existing_rows = existing_df.to_dict("records")
            print(f"Загружено {len(existing_rows)} строк из существующего CSV.")
        except Exception as e:
            print(f"Не удалось загрузить существующий CSV: {e}")

    all_rows = list(existing_rows)
    total = len(original_df)

    for idx, row in original_df.iterrows():
        review_id = row[ID_COL]

        if review_id in processed_ids:
            print(f"   Отзыв {review_id} уже обработан, пропускаем.")
            continue

        raw_comment = row[TEXT_COL]
        comment = (raw_comment.replace("\n", " ").replace("\r", " ")
                   if isinstance(raw_comment, str) else str(raw_comment))

        print(f"   -> Анализ {review_id} ({idx+1}/{total})...")
        analysis = analyze_single_review(review_id, comment, log_path)

        if analysis and "results" in analysis:
            # один отзыв = несколько строк (по числу извлечённых пар аспект/мнение)
            for item in analysis["results"]:
                new_row = {
                    ID_COL:        review_id,
                    TEXT_COL:      comment,
                    "aspect_term": item.get("aspect_term", ""),
                    "opinion":     item.get("opinion", ""),
                    "aspect":      item.get("aspect", ""),
                    "sentiment":   item.get("sentiment", ""),
                }
                for col in extra_cols:
                    new_row[col] = row[col]
                all_rows.append(new_row)
        else:
            # маркируем ERROR-строкой, чтобы потом легко отфильтровать и догнать вручную
            error_row = {
                ID_COL:        review_id,
                TEXT_COL:      comment,
                "aspect_term": "ERROR",
                "opinion":     "ERROR",
                "aspect":      "ERROR",
                "sentiment":   "ERROR",
            }
            for col in extra_cols:
                error_row[col] = row[col]
            all_rows.append(error_row)
            print(f"   Не удалось извлечь данные для отзыва {review_id}.")

        # сбрасываем CSV после каждого отзыва — медленнее, зато не теряем прогресс при падении
        _flush_csv(all_rows, output_path, extra_cols)

    print(f"\nГотово! Результат сохранён в {output_path}")
    print(f"Строк в итоговом файле: {len(all_rows)}")
    print(f"Подробный лог: {log_path}")


def _flush_csv(rows: list, output_path: str, extra_cols: list):
    if not rows:
        return
    df = pd.DataFrame(rows)
    desired = [ID_COL, TEXT_COL, "aspect_term", "opinion", "aspect", "sentiment"] + extra_cols
    actual  = [c for c in desired if c in df.columns]
    df[actual].to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        escapechar="\\",
    )


if __name__ == "__main__":
    process_csv_with_metadata(INPUT_CSV, OUTPUT_CSV, PROGRESS_LOG)

# Код для GPT-4.1 Mini

In [ ]:
import pandas as pd
import requests
import json
import time
import os
import re
import csv
from dotenv import load_dotenv

load_dotenv()

GENAPI_API_KEY = os.getenv("GENAPI_API_KEY")
if not GENAPI_API_KEY:
    raise ValueError("GENAPI_API_KEY не найден. Проверьте файл .env")

INPUT_CSV = "reviews.csv"
OUTPUT_CSV = "results_with_metadata_gpt_4.1_mini_netologia.csv"
SLEEP_BETWEEN_REQUESTS = 0.2

# openai-совместимый эндпоинт у gen-api (а не их кастомный /networks/...), отсюда и формат payload
URL = "https://proxy.gen-api.ru/v1/chat/completions"

ID_COL = "review_id"
TEXT_COL = "review_text"

SYSTEM_PROMPT = """Ты — аналитик отзывов об онлайн-курсах. Извлеки все мнения, аспектные термины, тональность и аспект строго по правилам. Работай строго по правилам ниже.

1.Точное цитирование — точные подстроки из review_text. Запрещен пересказ, синонимы, изменение формы слов.
2.Не пропускать оценочные фразы
3.Не извлекать лишние слова
4.Лучше разделить, чем объединить мнения
5. Опускать аспектное слово при извлечении мнения
6. Одно мнение = одна оценка = один аспектный термин = один аспект

ВХОД: review_id (int),  review_text (текст). Остальные поля игнорируй

ШАГ 1:
Аспектный термин — 1–4 слова, конкретный объект курса (лекции, задания, платформа, преподаватель и т.п.), к которому относится оценка.
—Выбирай минимальный: «сам контент» → «контент»
—Точная подстрока: ЗАПРЕЩЕНО изменять форму (падеж, число, время): «итоговую работу» НЕ равно «итоговая работа»
— НЕ местоимения, процессы/действия/абстракции без объекта, вводные фразы
— НОВЫЙ объект → новый аспектный термин

ШАГ 2:
Мнение — 2–10 слов, явная или косвенная оценка об объекте.
— Минимальная оценочная часть фразы с модификатором («очень», «довольно») при его наличии
— Точная подстрока: нельзя добавлять, удалять, обобщать, переформулировать слова
— Нельзя дублировать аспектное слово
— В пределах одного предложения
— Считать мнением: оценочные прилагательные/наречия/глаголы, устойчивые выражения («на высшем уровне»), скрытые оценки («длится 7 минут»), развернутые оценки («сформировался четкий план моего краткосрочного и долгосрочного развития»)
— НЕ считать мнением: нейтральные факты («было 12 модулей»), перечисления  тем/форматов без оценки.

РАЗДЕЛЯТЬ:
1.противопоставления (но, однако, зато): («материал хороший, но неструктурированный → «хороший» и «неструктурированный»
2.разные атрибуты одного объекта: «проверяют быстро, с полезными комментариями» → «быстро» (скорость) и «с полезными комментариями» (качество).
3.разные аспекты в одной фразе (например, Стоимость + Контент): «купила за 50 тыс руб, а за час прошла курс» → «купила за 50 тыс руб» (Цена); «за час прошла курс» (Контент)

НЕ РАЗДЕЛЯТЬ:
1.однородные члены без противопоставления с одним и тем же атрибутом объекта («отношение сотрудников и наставников очень доброе, понимающее→описано только качество отношения => единое мнение «очень доброе, понимающее»)

САМОПРОВЕРКА
— Аспектный термин: точная подстрока (1–4 слова), конкретный объект.
— Мнение: точная подстрока с оценкой.
— Все оценки извлечены, нет выдуманных, нет дублей.

ШАГ 3:
ТОНАЛЬНОСТЬ:
положительная — положительная оценка
отрицательная — отрицательная оценка
АСПЕКТЫ:
Преподаватели — работа преподавателей и лекторов (стиль подачи материала, компетентность, умение объяснять, вовлеченность, готовность отвечать на вопросы по содержанию)
Контент — теоретическое содержание курса (программа, темы, структура, полнота, актуальность, качество, объем информации, последовательность разделов/модулей, формат, необходимость поиска дополнительных материалов)
Практика — проекты, кейсы, задания и т.п., т.е. все что студент выполняет для отработки навыков (сложность, полезность, количество, наличие, связь с теорией)
Техническая поддержка — качество помощи от технической поддержки (скорость ответов, доброжелательность, компетентность)
Команда сопровождения — работа кураторов / координаторов (качество и наличие обратной связи по заданиям, скорость проверки, доброжелательность, решение организационных и учебных вопросов)
Сообщество – общение с одногруппниками, чаты, взаимопомощь, атмосфера в коллективе
Платформа — интерфейс, навигация, стабильность работы сайта/приложения
Стоимость — стоимость, скидки, рассрочка, соотношение цена/качество, возврат средств
Результаты — приобретенные после окончания курса навыки, трудоустройство после окончания курса, достижения после окончания курса, карьерный рост, портфолио
Сертификаты — наличие диплома или сертификата после обучения (ценность, процесс получения)
Гибкость учебного процесса — график, дедлайны, наличие записей лекций, возможность совмещать с работой и личной жизнью возможность обучаться в любое удобное время, возможность продления дедлайна
Дополнительно — бонусы, плюшки, стажировки, дополнительные активности, внеучебные активности, программа трудоустройства, составление резюме
Общее — общее впечатление без указания конкретных причин, если нельзя отнести к другим аспектам
3. Выбор аспекта
— Сначала ориентируйся на аспектный термин, затем на мнение
— Наиболее конкретный аспект

ФОРМАТ ВЫВОДА JSON
{
 "review_id": <id>,
 "results": [
 {
 "aspect_term": "точная подстрока из текста (1–4 слова)",
 "opinion": "точная подстрока из текста (оценочная часть)",
 "sentiment": "положительная | отрицательная ",
 "aspect": "из списка аспектов"
 }
 ]
}

ПРИМЕРЫ
[
 {
 "input": {
 "review_id": 3,
 " review_text": "Кураторы тоже большие молодцы — проверяют работы быстро, с конкретными и полезными комментариями."
 },
 "output": {
 "review_id": 3,
 "results": [
 {"aspect_term": "кураторы", "opinion": "большие молодцы", "sentiment": "положительная", "aspect": "Команда сопровождения"},
 {"aspect_term": "кураторы", "opinion": "проверяют работы быстро", "sentiment": "положительная", "aspect": "Команда сопровождения"},
 {"aspect_term": "кураторы", "opinion": "с конкретными и полезными комментариями", "sentiment": "положительная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 2,
 " review_text": "внутренняя атмосфера тепла, уюта, очень уважительного человеческого отношения!"
 },
 "output": {
 "review_id": 2,
 "results": [
 {"aspect_term": "внутренняя атмосфера", "opinion": "тепла, уюта, очень уважительного человеческого отношения", "sentiment": "положительная", "aspect": "Общее"}
 ]
 }
 },
 {
 "input": {
 "review_id": 30,
 " review_text": "На уроке «КАК повысить конверсию», нам рассказывают: ЧТО такое конверсия. Хочу отметить, что урок длится 7 минут."
 },
 "output": {
 "review_id": 30,
 "results": [
 {"aspect_term": "уроке «КАК повысить конверсию»", "opinion": "рассказывают: ЧТО такое конверсия", "sentiment": "отрицательная", "aspect": "Контент"}
 {"aspect_term": "урок", "opinion": "длится 7 минут", "sentiment": "отрицательная", "aspect": "Контент"}
 ]
 }
 },
 {
 "input": {
 "review_id": 6,
 " review_text": "Качественные материалы составлены таким образом, что вы не чувствуете обязанности пройти урок, вы испытываете удовольствие от получения новых знаний."
 },
 "output": {
 "review_id": 6,
 "results": [
 {"aspect_term": "материалы", "opinion": "качественные", "sentiment": "положительная", "aspect": "Контент"},
 {"aspect_term": "материалы", "opinion": "испытываете удовольствие от получения новых знаний", "sentiment": "положительная", "aspect": "Контент"}
 ]
 }
 },
 {
 "input": {
 "review_id": 14,
 " review_text": "Куратор не помощник, только отписчик."
 },
 "output": {
 "review_id": 14,
 "results": [
 {"aspect_term": "куратор", "opinion": "не помощник, только отписчик", "sentiment": "отрицательная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 13,
 " review_text": "Технические сбои бывают, но решаются быстро."
 },
 "output": {
 "review_id": 13,
 "results": [
 {"aspect_term": "технические сбои", "opinion": "бывают", "sentiment": "отрицательная", "aspect": "Платформа"},
 {"aspect_term": "технические сбои", "opinion": "решаются быстро", "sentiment": "положительная", "aspect": "Платформа"}
 ]
 }
 },
 {
 "input": {
 "review_id": 23,
 " review_text": "Отношение сотрудников и наставников очень доброе, понимающее и профессиональное."
 },
 "output": {
 "review_id": 23,
 "results": [
 {"aspect_term": "отношение сотрудников и наставников", "opinion": "очень доброе, понимающее и профессиональное", "sentiment": "положительная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 17,
 " review_text": "Теперь у меня есть диплом гос образца о профессиональной переподготовке."
 },
 "output": {
 "review_id": 17,
 "results": [
 {"aspect_term": "диплом гос образца", "opinion": "у меня есть", "sentiment": "положительная", "aspect": "Сертификаты"}
 ]
 }
 },
 {
 "input": {
 "review_id": 4,
 " review_text": "Лекции были примерно такие «Если сложить 2+2, то в итоге будет 4», в домашке к этой лекции было «Соберите ракету и отправьте ее в космос»."
 },
 "output": {
 "review_id": 4,
 "results": [
 {"aspect_term": "лекции", "opinion": " «Если сложить 2+2, то в итоге будет 4»", "sentiment": "отрицательная", "aspect": "Контент"},
 {"aspect_term": "домашке", "opinion": "было «Соберите ракету и отправьте ее в космос»", "sentiment": "отрицательная", "aspect": "Практика"}
 ]
 }
 },
 {
 "input": {
 "review_id": 12,
 " review_text": "Практика проектов по нейросетям проходила на учебной платформе, которая периодически зависала, или в приложении, где стабильность работы тоже оставляла желать лучшего."
 },
 "output": {
 "review_id": 12,
 "results": [
 {"aspect_term": "учебной платформе", "opinion": "периодически зависала", "sentiment": "отрицательная", "aspect": "Платформа"},
 {"aspect_term": "приложении", "opinion": "стабильность работы тоже оставляла желать лучшего", "sentiment": "отрицательная", "aspect": "Платформа"}
 ]
 }
 },
]
"""


def extract_json_from_text(text):
    # модель иногда дописывает пояснения до/после JSON, поэтому regex-ом вытаскиваем фигурные скобки
    if not text:
        return None
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None
    return None


def analyze_single_review(review_id, comment_text):
    payload = {
        "model": "gpt-4.1-mini",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"review_id: {review_id}\ncomment: {comment_text}"}
        ],
        "temperature": 0.1,
        "max_tokens": 6000,
        "response_format": {"type": "json_object"}
    }

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {GENAPI_API_KEY}"
    }

    try:
        # таймаут с запасом — на длинных отзывах модель думает дольше обычного
        response = requests.post(URL, headers=headers, json=payload, timeout=120)
        response.raise_for_status()
        result_json = response.json()
        answer_text = result_json['choices'][0]['message']['content']

        parsed = extract_json_from_text(answer_text)
        if parsed:
            return parsed
        else:
            print(f"   Не удалось извлечь JSON из ответа для отзыва {review_id}")
            print(f"   Ответ модели: {answer_text[:200]}...")
            return None
    except Exception as e:
        print(f"   Ошибка при анализе отзыва {review_id}: {e}")
        return None


def process_csv_with_metadata(input_path, output_path):
    try:
        # QUOTE_ALL + escapechar — иначе CSV с кавычками внутри отзывов разъезжается
        original_df = pd.read_csv(
            input_path,
            quoting=csv.QUOTE_ALL,
            escapechar='\\',
            on_bad_lines='skip'
        )
        print(f"Загружено {len(original_df)} отзывов.")
        print(f"Найдены колонки: {list(original_df.columns)}")
    except FileNotFoundError:
        print(f"Файл {input_path} не найден.")
        return

    if ID_COL not in original_df.columns:
        raise ValueError(f"Колонка '{ID_COL}' не найдена в CSV.")
    if TEXT_COL not in original_df.columns:
        raise ValueError(f"Колонка '{TEXT_COL}' не найдена в CSV.")

    # всё, что не id и не текст — переносим в результат как есть (рейтинг, источник, дата и т.п.)
    extra_cols = [col for col in original_df.columns if col not in [ID_COL, TEXT_COL]]
    print(f"Дополнительные колонки (будут сохранены): {extra_cols}")

    all_rows = []
    total = len(original_df)

    for idx, row in original_df.iterrows():
        review_id = row[ID_COL]
        raw_comment = row[TEXT_COL]
        if isinstance(raw_comment, str):
            comment = raw_comment.replace('\n', ' ').replace('\r', ' ')
        else:
            comment = raw_comment

        print(f"   -> Анализ отзыва {review_id} ({idx+1}/{total})...")
        analysis = analyze_single_review(review_id, comment)

        if analysis and 'results' in analysis:
            # один отзыв = несколько строк (по числу извлечённых пар аспект/мнение)
            for item in analysis['results']:
                new_row = {
                    ID_COL: review_id,
                    TEXT_COL: comment,
                    'aspect_term': item.get('aspect_term', ''),
                    'opinion': item.get('opinion', ''),
                    'aspect': item.get('aspect', ''),
                    'sentiment': item.get('sentiment', '')
                }
                for col in extra_cols:
                    new_row[col] = row[col]
                all_rows.append(new_row)
        else:
            # маркируем ERROR-строкой, чтобы потом легко отфильтровать и догнать руками
            error_row = {
                ID_COL: review_id,
                TEXT_COL: comment,
                'aspect_term': 'ERROR',
                'opinion': 'ERROR',
                'aspect': 'ERROR',
                'sentiment': 'ERROR'
            }
            for col in extra_cols:
                error_row[col] = row[col]
            all_rows.append(error_row)
            print(f"   Не удалось извлечь данные для отзыва {review_id}.")

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if not all_rows:
        print("Нет данных для сохранения.")
        return

    result_df = pd.DataFrame(all_rows)

    # порядок колонок: ключевые поля впереди, потом всё остальное из исходника
    desired_order = [ID_COL, TEXT_COL, 'aspect_term', 'opinion', 'aspect', 'sentiment'] + extra_cols
    actual_order = [col for col in desired_order if col in result_df.columns]
    result_df = result_df[actual_order]

    result_df.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        escapechar='\\'
    )
    print(f"Готово! Результат сохранён в {output_path}")
    print(f"Всего строк в итоговом файле: {len(result_df)}")


if __name__ == "__main__":
    process_csv_with_metadata(INPUT_CSV, OUTPUT_CSV)

# Код для YandexGPT 5.1 Pro

In [ ]:
import pandas as pd
import requests
import json
import time
import os
import re
import csv
from dotenv import load_dotenv

load_dotenv()

YANDEX_API_KEY = os.getenv("YANDEX_API_KEY")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
if not YANDEX_API_KEY or not FOLDER_ID:
    raise ValueError("API ключ или Folder ID не найдены. Проверьте файл .env")

INPUT_CSV = "reviews.csv"
OUTPUT_CSV = "results_with_metadata_yandex.csv"
SLEEP_BETWEEN_REQUESTS = 0.2

URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

ID_COL = "review_id"
TEXT_COL = "review_text"

SYSTEM_PROMPT = """Ты — аналитик отзывов об онлайн-курсах. Извлеки все мнения, аспектные термины, тональность и аспект строго по правилам. Работай строго по правилам ниже.

1.Точное цитирование — точные подстроки из review_text. Запрещен пересказ, синонимы, изменение формы слов.
2.Не пропускать оценочные фразы
3.Не извлекать лишние слова
4.Лучше разделить, чем объединить мнения
5. Опускать аспектное слово при извлечении мнения
6. Одно мнение = одна оценка = один аспектный термин = один аспект

ВХОД: review_id (int),  review_text (текст). Остальные поля игнорируй

ШАГ 1:
Аспектный термин — 1–4 слова, конкретный объект курса (лекции, задания, платформа, преподаватель и т.п.), к которому относится оценка.
—Выбирай минимальный: «сам контент» → «контент»
—Точная подстрока: ЗАПРЕЩЕНО изменять форму (падеж, число, время): «итоговую работу» НЕ равно «итоговая работа»
— НЕ местоимения, процессы/действия/абстракции без объекта, вводные фразы
— НОВЫЙ объект → новый аспектный термин

ШАГ 2:
Мнение — 2–10 слов, явная или косвенная оценка об объекте.
— Минимальная оценочная часть фразы с модификатором («очень», «довольно») при его наличии
— Точная подстрока: нельзя добавлять, удалять, обобщать, переформулировать слова
— Нельзя дублировать аспектное слово
— В пределах одного предложения
— Считать мнением: оценочные прилагательные/наречия/глаголы, устойчивые выражения («на высшем уровне»), скрытые оценки («длится 7 минут»), развернутые оценки («сформировался четкий план моего краткосрочного и долгосрочного развития»)
— НЕ считать мнением: нейтральные факты («было 12 модулей»), перечисления  тем/форматов без оценки.

РАЗДЕЛЯТЬ:
1.противопоставления (но, однако, зато): («материал хороший, но неструктурированный → «хороший» и «неструктурированный»
2.разные атрибуты одного объекта: «проверяют быстро, с полезными комментариями» → «быстро» (скорость) и «с полезными комментариями» (качество).
3.разные аспекты в одной фразе (например, Стоимость + Контент): «купила за 50 тыс руб, а за час прошла курс» → «купила за 50 тыс руб» (Цена); «за час прошла курс» (Контент)

НЕ РАЗДЕЛЯТЬ:
1.однородные члены без противопоставления с одним и тем же атрибутом объекта («отношение сотрудников и наставников очень доброе, понимающее→описано только качество отношения => единое мнение «очень доброе, понимающее»)

САМОПРОВЕРКА
— Аспектный термин: точная подстрока (1–4 слова), конкретный объект.
— Мнение: точная подстрока с оценкой.
— Все оценки извлечены, нет выдуманных, нет дублей.

ШАГ 3:
ТОНАЛЬНОСТЬ:
положительная — положительная оценка
отрицательная — отрицательная оценка
АСПЕКТЫ:
Преподаватели — работа преподавателей и лекторов (стиль подачи материала, компетентность, умение объяснять, вовлеченность, готовность отвечать на вопросы по содержанию)
Контент — теоретическое содержание курса (программа, темы, структура, полнота, актуальность, качество, объем информации, последовательность разделов/модулей, формат, необходимость поиска дополнительных материалов)
Практика — проекты, кейсы, задания и т.п., т.е. все что студент выполняет для отработки навыков (сложность, полезность, количество, наличие, связь с теорией)
Техническая поддержка — качество помощи от технической поддержки (скорость ответов, доброжелательность, компетентность)
Команда сопровождения — работа кураторов / координаторов (качество и наличие обратной связи по заданиям, скорость проверки, доброжелательность, решение организационных и учебных вопросов)
Сообщество – общение с одногруппниками, чаты, взаимопомощь, атмосфера в коллективе
Платформа — интерфейс, навигация, стабильность работы сайта/приложения
Стоимость — стоимость, скидки, рассрочка, соотношение цена/качество, возврат средств
Результаты — приобретенные после окончания курса навыки, трудоустройство после окончания курса, достижения после окончания курса, карьерный рост, портфолио
Сертификаты — наличие диплома или сертификата после обучения (ценность, процесс получения)
Гибкость учебного процесса — график, дедлайны, наличие записей лекций, возможность совмещать с работой и личной жизнью возможность обучаться в любое удобное время, возможность продления дедлайна
Дополнительно — бонусы, плюшки, стажировки, дополнительные активности, внеучебные активности, программа трудоустройства, составление резюме
Общее — общее впечатление без указания конкретных причин, если нельзя отнести к другим аспектам
3. Выбор аспекта
— Сначала ориентируйся на аспектный термин, затем на мнение
— Наиболее конкретный аспект

ФОРМАТ ВЫВОДА JSON
{
 "review_id": <id>,
 "results": [
 {
 "aspect_term": "точная подстрока из текста (1–4 слова)",
 "opinion": "точная подстрока из текста (оценочная часть)",
 "sentiment": "положительная | отрицательная ",
 "aspect": "из списка аспектов"
 }
 ]
}

ПРИМЕРЫ
[
 {
 "input": {
 "review_id": 3,
 " review_text": "Кураторы тоже большие молодцы — проверяют работы быстро, с конкретными и полезными комментариями."
 },
 "output": {
 "review_id": 3,
 "results": [
 {"aspect_term": "кураторы", "opinion": "большие молодцы", "sentiment": "положительная", "aspect": "Команда сопровождения"},
 {"aspect_term": "кураторы", "opinion": "проверяют работы быстро", "sentiment": "положительная", "aspect": "Команда сопровождения"},
 {"aspect_term": "кураторы", "opinion": "с конкретными и полезными комментариями", "sentiment": "положительная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 2,
 " review_text": "внутренняя атмосфера тепла, уюта, очень уважительного человеческого отношения!"
 },
 "output": {
 "review_id": 2,
 "results": [
 {"aspect_term": "внутренняя атмосфера", "opinion": "тепла, уюта, очень уважительного человеческого отношения", "sentiment": "положительная", "aspect": "Общее"}
 ]
 }
 },
 {
 "input": {
 "review_id": 30,
 " review_text": "На уроке «КАК повысить конверсию», нам рассказывают: ЧТО такое конверсия. Хочу отметить, что урок длится 7 минут."
 },
 "output": {
 "review_id": 30,
 "results": [
 {"aspect_term": "уроке «КАК повысить конверсию»", "opinion": "рассказывают: ЧТО такое конверсия", "sentiment": "отрицательная", "aspect": "Контент"}
 {"aspect_term": "урок", "opinion": "длится 7 минут", "sentiment": "отрицательная", "aspect": "Контент"}
 ]
 }
 },
 {
 "input": {
 "review_id": 6,
 " review_text": "Качественные материалы составлены таким образом, что вы не чувствуете обязанности пройти урок, вы испытываете удовольствие от получения новых знаний."
 },
 "output": {
 "review_id": 6,
 "results": [
 {"aspect_term": "материалы", "opinion": "качественные", "sentiment": "положительная", "aspect": "Контент"},
 {"aspect_term": "материалы", "opinion": "испытываете удовольствие от получения новых знаний", "sentiment": "положительная", "aspect": "Контент"}
 ]
 }
 },
 {
 "input": {
 "review_id": 14,
 " review_text": "Куратор не помощник, только отписчик."
 },
 "output": {
 "review_id": 14,
 "results": [
 {"aspect_term": "куратор", "opinion": "не помощник, только отписчик", "sentiment": "отрицательная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 13,
 " review_text": "Технические сбои бывают, но решаются быстро."
 },
 "output": {
 "review_id": 13,
 "results": [
 {"aspect_term": "технические сбои", "opinion": "бывают", "sentiment": "отрицательная", "aspect": "Платформа"},
 {"aspect_term": "технические сбои", "opinion": "решаются быстро", "sentiment": "положительная", "aspect": "Платформа"}
 ]
 }
 },
 {
 "input": {
 "review_id": 23,
 " review_text": "Отношение сотрудников и наставников очень доброе, понимающее и профессиональное."
 },
 "output": {
 "review_id": 23,
 "results": [
 {"aspect_term": "отношение сотрудников и наставников", "opinion": "очень доброе, понимающее и профессиональное", "sentiment": "положительная", "aspect": "Команда сопровождения"}
 ]
 }
 },
 {
 "input": {
 "review_id": 17,
 " review_text": "Теперь у меня есть диплом гос образца о профессиональной переподготовке."
 },
 "output": {
 "review_id": 17,
 "results": [
 {"aspect_term": "диплом гос образца", "opinion": "у меня есть", "sentiment": "положительная", "aspect": "Сертификаты"}
 ]
 }
 },
 {
 "input": {
 "review_id": 4,
 " review_text": "Лекции были примерно такие «Если сложить 2+2, то в итоге будет 4», в домашке к этой лекции было «Соберите ракету и отправьте ее в космос»."
 },
 "output": {
 "review_id": 4,
 "results": [
 {"aspect_term": "лекции", "opinion": " «Если сложить 2+2, то в итоге будет 4»", "sentiment": "отрицательная", "aspect": "Контент"},
 {"aspect_term": "домашке", "opinion": "было «Соберите ракету и отправьте ее в космос»", "sentiment": "отрицательная", "aspect": "Практика"}
 ]
 }
 },
 {
 "input": {
 "review_id": 12,
 " review_text": "Практика проектов по нейросетям проходила на учебной платформе, которая периодически зависала, или в приложении, где стабильность работы тоже оставляла желать лучшего."
 },
 "output": {
 "review_id": 12,
 "results": [
 {"aspect_term": "учебной платформе", "opinion": "периодически зависала", "sentiment": "отрицательная", "aspect": "Платформа"},
 {"aspect_term": "приложении", "opinion": "стабильность работы тоже оставляла желать лучшего", "sentiment": "отрицательная", "aspect": "Платформа"}
 ]
 }
 },
]
"""


def extract_json_from_text(text):
    if not text:
        return None
    # на случай, если модель обернула json в комментарии — выдёргиваем фигурные скобки
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None
    return None


def analyze_single_review(review_id, comment_text):
    prompt = {
        # /rc = release candidate, у яндекса так помечены свежие версии моделей
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/rc",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            # maxTokens у яндекса именно строкой, не числом — иначе валится валидация
            "maxTokens": "6000"
        },
        "messages": [
            {"role": "system", "text": SYSTEM_PROMPT},
            {"role": "user", "text": f"review_id: {review_id}\ncomment: {comment_text}"}
        ]
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {YANDEX_API_KEY}"
    }
    try:
        response = requests.post(URL, headers=headers, json=prompt, timeout=60)
        response.raise_for_status()
        result_json = response.json()
        # путь до контента у яндекса свой: result -> alternatives[0] -> message -> text
        answer_text = result_json['result']['alternatives'][0]['message']['text']

        parsed = extract_json_from_text(answer_text)
        if parsed:
            return parsed
        else:
            print(f"   Не удалось извлечь JSON из ответа для отзыва {review_id}")
            print(f"   Ответ модели: {answer_text[:200]}...")
            return None
    except Exception as e:
        print(f"   Ошибка при анализе отзыва {review_id}: {e}")
        return None


def process_csv_with_metadata(input_path, output_path):
    try:
        original_df = pd.read_csv(
            input_path,
            quoting=csv.QUOTE_ALL,
            escapechar='\\',
            on_bad_lines='skip'
        )
        print(f"Загружено {len(original_df)} отзывов.")
        print(f"Найдены колонки: {list(original_df.columns)}")
    except FileNotFoundError:
        print(f"Файл {input_path} не найден.")
        return

    if ID_COL not in original_df.columns:
        raise ValueError(f"Колонка '{ID_COL}' не найдена в CSV.")
    if TEXT_COL not in original_df.columns:
        raise ValueError(f"Колонка '{TEXT_COL}' не найдена в CSV.")

    extra_cols = [col for col in original_df.columns if col not in [ID_COL, TEXT_COL]]
    print(f"Дополнительные колонки (будут сохранены): {extra_cols}")

    all_rows = []
    total = len(original_df)

    for idx, row in original_df.iterrows():
        review_id = row[ID_COL]
        raw_comment = row[TEXT_COL]
        # переносы внутри текста потом ломают итоговый CSV, поэтому схлопываем здесь
        if isinstance(raw_comment, str):
            comment = raw_comment.replace('\n', ' ').replace('\r', ' ')
        else:
            comment = raw_comment

        print(f"   -> Анализ отзыва {review_id} ({idx+1}/{total})...")
        analysis = analyze_single_review(review_id, comment)

        if analysis and 'results' in analysis:
            for item in analysis['results']:
                new_row = {
                    ID_COL: review_id,
                    TEXT_COL: comment,
                    'aspect_term': item.get('aspect_term', ''),
                    'opinion': item.get('opinion', ''),
                    'aspect': item.get('aspect', ''),
                    'sentiment': item.get('sentiment', '')
                }
                for col in extra_cols:
                    new_row[col] = row[col]
                all_rows.append(new_row)
        else:
            # ERROR-строка вместо пропуска — чтобы потом отфильтровать grep-ом и перегнать руками
            error_row = {
                ID_COL: review_id,
                TEXT_COL: comment,
                'aspect_term': 'ERROR',
                'opinion': 'ERROR',
                'aspect': 'ERROR',
                'sentiment': 'ERROR'
            }
            for col in extra_cols:
                error_row[col] = row[col]
            all_rows.append(error_row)
            print(f"   Не удалось извлечь данные для отзыва {review_id}.")

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if not all_rows:
        print("Нет данных для сохранения.")
        return

    result_df = pd.DataFrame(all_rows)

    desired_order = [ID_COL, TEXT_COL, 'aspect_term', 'opinion', 'aspect', 'sentiment'] + extra_cols
    actual_order = [col for col in desired_order if col in result_df.columns]
    result_df = result_df[actual_order]

    result_df.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        escapechar='\\'
    )
    print(f"Готово! Результат сохранён в {output_path}")
    print(f"Всего строк в итоговом файле: {len(result_df)}")


if __name__ == "__main__":
    process_csv_with_metadata(INPUT_CSV, OUTPUT_CSV)